# 47. Joslim's code in US-Net mode, and Joslim's own Pareto set

Two controls on notebook 39, where Joslim (`joslim_r50`) read 74.02 mean
top-1, about 2.8 points under our US-Net A (76.86), and under A at width 1.0
as well. Same protocol as notebook 39: CIFAR-100 32x32, CIFAR ResNet-50, 100
epochs, batch 256, SGD nesterov lr 0.2 cosine with 5 warm-up epochs,
wd 5e-4, widths 0.25-1.00, BN recalibrated on 20 batches per width.

| | GPU | what | code |
|---|---|---|---|
| slim_r50 | 0 | Joslim's code in its "Slim" mode, the paper's US-Net baseline: `--slim --slim_uniform --tau 1` in place of `--tau 195 --prior_points 20`, every other flag as joslim_r50. Each step trains full width, the smallest width and two uniform widths from U(0.25, 1), the last three distilled from the full-width logits. Then the sixteen-width evaluation joslim_r50 had | enyac-group/Joslim at 9b743d9 + `third_party/patches/joslim.patch` |
| joslim_r50 Pareto | 1 | No training. Joslim's own evaluation of the joslim_r50 checkpoint from notebook 39 (`eval_checkpoints.py` without `--uniform`): the non-dominated per-layer width configurations visited in training, each BN-recalibrated and tested, accuracy against FLOPs | same |

The Pareto evaluation crashed in notebook 39 under current numpy (the
per-layer widths are ragged and were turned into an ndarray);
`joslim.patch` now selects them with the mask in plain Python, see
`third_party/patches/joslim_README.md`.

## Inputs

Attach the CIFAR-100 dataset and the output of notebook 39, which holds
`ckpt/joslim_r50.pt`. Without that checkpoint the Pareto job is skipped
with a message and slim_r50 trains anyway.

## More than one session

slim_r50 took ~4.3 h on an RTX 5060 Ti (joslim_README.md, Timing), and a
T4 is 2-3x slower in fp32: expect ~10-13 h, so one resume. It checkpoints
every epoch. When the session ends, Save Version, then in a new run attach
that version's output (pick the version by number, not Latest) together
with notebook 39's output and run again: slim_r50 continues from its
checkpoint and is evaluated when it finishes. The Pareto job takes about
half a minute per configuration and runs again in every session next to
the training; the repetition is harmless.

In [ ]:
# Fixed for this notebook.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'
# key: (repository, pinned commit, folder, patch, extra pip packages)
UPSTREAMS = {'joslim': ('https://github.com/enyac-group/Joslim.git', '9b743d9b50ade9fc2cd33d255dfbb936e4894733', 'Joslim', 'joslim.patch', 'botorch==0.16.1 gpytorch==1.15.2 linear_operator==0.6.1')}
WORKER_ENV = {'JOSLIM_WORKERS': '2', 'LCS_WORKERS': '2'}

In [ ]:
import os, queue, re, shutil, subprocess, sys, threading, time
import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
if n_gpu < 2:
    raise SystemExit('Needs GPU T4 x2: one card per method.')

WORK = '/kaggle/working'
OURS = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(OURS):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, OURS],
                   check=True)
print('our code at', subprocess.run(
    ['git', '-C', OURS, 'log', '-1', '--format=%h %s'],
    capture_output=True, text=True).stdout.strip())

CODE = {}
packages = []
for key, (url, commit, folder, patch, pip) in UPSTREAMS.items():
    CODE[key] = os.path.join(WORK, folder)
    if not os.path.isdir(CODE[key]):
        subprocess.run(['git', 'clone', url, CODE[key]], check=True)
        subprocess.run(['git', '-C', CODE[key], 'checkout', '-q', commit],
                       check=True)
        subprocess.run(['git', '-C', CODE[key], 'apply', os.path.join(
            OURS, 'third_party', 'patches', patch)], check=True)
    print(key, 'at', subprocess.run(
        ['git', '-C', CODE[key], 'log', '-1', '--format=%h %ad',
         '--date=short'], capture_output=True, text=True).stdout.strip(),
        '+', patch)
    packages += pip.split()

# install without letting pip replace Kaggle's torch
base = torch.__version__.split('+')[0]
with open(os.path.join(WORK, 'constraints.txt'), 'w') as handle:
    handle.write('torch=={}\n'.format(base))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                os.path.join(WORK, 'constraints.txt')] + packages, check=True)

In [ ]:
# the folder that contains cifar-100-python/, or a place to download it
DATA = None
for root, dirs, _ in os.walk('/kaggle/input', followlinks=True):
    if 'cifar-100-python' in dirs:
        DATA = root
        break
if DATA is None:
    DATA = os.path.join(WORK, 'data')
    os.makedirs(DATA, exist_ok=True)
    print('no CIFAR-100 attached: it will be downloaded into', DATA)
else:
    print('CIFAR-100 from', DATA)

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs):
    """one command per card, output tagged and interleaved"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for label, gpu, cwd, command in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, **WORKER_ENV)
        proc = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] gpu {}: {}'.format(label, gpu, ' '.join(command)),
              flush=True)
    started, remaining = time.time(), len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[round(float(width), 2)] = (
                float(loss), float(top1))
        print('[{}] {}'.format(label, line), flush=True)
    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
CK = os.path.join(WORK, 'ckpt')
os.makedirs(CK, exist_ok=True)
# slim_r50 resumes from ckpt/slim_r50.pt (an earlier session of this notebook);
# the Pareto job reads ckpt/joslim_r50.pt (the output of notebook 39)
for name in ['slim_r50.pt', 'joslim_r50.pt']:
    target = os.path.join(CK, name)
    if not os.path.exists(target):
        for root, _, files in os.walk('/kaggle/input', followlinks=True):
            if name in files:
                shutil.copy(os.path.join(root, name), target)
                print('restored', name, 'from', root)
                break
print('slim_r50', 'resumes from its checkpoint'
      if os.path.exists(os.path.join(CK, 'slim_r50.pt'))
      else 'starts at epoch 1')
HAVE_JOSLIM = os.path.exists(os.path.join(CK, 'joslim_r50.pt'))
if HAVE_JOSLIM:
    print('joslim_r50 checkpoint found: its Pareto set is evaluated on GPU 1')
else:
    print('NO joslim_r50.pt under /kaggle/input: attach the output of '
          'notebook 39. The Pareto job is skipped; slim_r50 trains anyway.')

In [ ]:
JOSLIM = ['--dataset', 'CIFAR100', '--datapath', DATA,
          '--network', 'slim_resnet50_cifar', '--epochs', '100',
          '--warmup', '5', '--baselr', '0.2', '--scheduler', 'cosine_decay',
          '--batch_size', '256', '--wd', '5e-4', '--mmt', '0.9',
          '--nesterov', '--label_smoothing', '0', '--lower_channel', '0.25',
          '--num_sampled_arch', '2', '--baseline', '-3',
          '--print_freq', '100', '--ckpt_dir', CK,
          '--slim', '--slim_uniform', '--tau', '1']
# the evaluation flags of notebook 39, without --name
EVAL = ['--dataset', 'CIFAR100', '--datapath', DATA,
        '--network', 'slim_resnet50_cifar', '--batch_size', '256',
        '--lower_channel', '0.25', '--ckpt_dir', CK]
jobs = [('slim_r50', '0', CODE['joslim'],
         [sys.executable, '-u', 'joslim.py', '--name', 'slim_r50'] + JOSLIM)]
if HAVE_JOSLIM:
    # Joslim's own evaluation (no --uniform): its Pareto set over per-layer
    # widths, printed as "(i/n) Acc: top1 top5, MFLOPs: x (y %)"
    jobs.append(('joslim_r50_pareto', '1', CODE['joslim'],
                 [sys.executable, '-u', 'eval_checkpoints.py',
                  '--name', 'joslim_r50'] + EVAL))
codes = run_pinned(jobs)
print(codes)

In [ ]:
# slim_r50 at the sixteen uniform widths, the evaluation joslim_r50 had
if codes.get('slim_r50') == 0:
    run_pinned([('slim_r50', '0', CODE['joslim'],
                 [sys.executable, '-u', 'eval_checkpoints.py',
                  '--name', 'slim_r50'] + EVAL
                 + ['--uniform', '--tag', 'slim_r50'])])
else:
    print('slim_r50 training exited with code', codes.get('slim_r50'),
          '- not evaluated')

In [ ]:
# Joslim's Pareto set for joslim_r50 (per-layer widths), as a table
import numpy as np
FULL_MFLOPS = 1298.0  # CIFAR ResNet-50, conv MACs (joslim_README.md)
path = os.path.join(CODE['joslim'], 'results', 'joslim_r50_eval_pareto.txt')
if not os.path.exists(path):
    print('no Pareto results: the job was skipped or failed, see its '
          '[joslim_r50_pareto] lines above')
else:
    shutil.copy(path, os.path.join(WORK, 'joslim_r50_eval_pareto.txt'))
    flops, top1, top5 = np.loadtxt(path, ndmin=2)
    print('joslim_r50, Pareto set (smallest and full width included)')
    print('{:>9}{:>9}{:>8}{:>8}'.format('FLOPs %', 'MFLOPs', 'top-1',
                                        'top-5'))
    for ratio, one, five in zip(flops, top1, top5):
        print('{:>9.2f}{:>9.1f}{:>8.2f}{:>8.2f}'.format(
            100.0 * ratio, FULL_MFLOPS * ratio, one, five))

## Results, against A (BX, 76.86) and K (DD, 77.50)

Send back this table and the `val ... -1/100` lines above it.

How to read slim_r50. It is US-Net trained by Joslim's code with our
patch, on A's recipe. If it reaches about A's level (around 77 or more at
width 1.0, where A has 78.04), Joslim's codebase and our patch are fine and
Joslim's gap is the method. If it also lands around 75, the codebase or the
patch is suspect, and joslim_r50 cannot be read as Joslim. A gap well under
a point can come from the recipe details Joslim's code keeps
(joslim_README.md, deviations 6-8: weight-decay groups, initialisation,
1x1-conv classifier). The `WM: ... MFLOPs` lines of the slim_r50 evaluation
give the FLOPs of each uniform width, to set against the Pareto table
above.

In [ ]:
A_REF = {0.25: 75.26, 0.3: 75.35, 0.35: 75.54, 0.4: 76.01, 0.45: 76.3, 0.5: 76.77, 0.55: 76.63, 0.6: 77.03, 0.65: 77.13, 0.7: 77.2, 0.75: 77.39, 0.8: 77.56, 0.85: 77.6, 0.9: 77.99, 0.95: 78.01, 1.0: 78.04}
K_REF = {0.25: 75.63, 0.3: 76.29, 0.35: 76.54, 0.4: 76.97, 0.45: 77.67, 0.5: 77.43, 0.55: 77.58, 0.6: 77.62, 0.65: 77.83, 0.7: 78.02, 0.75: 78.0, 0.8: 78.01, 0.85: 78.09, 0.9: 78.23, 0.95: 78.08, 1.0: 77.95}
labels = ['slim_r50']
print('{:>7}{:>8}{:>8}'.format('width', 'A', 'K') + ''.join(
    '{:>12}{:>7}{:>7}'.format(b[:11], 'vs A', 'vs K') for b in labels))
for width in sorted(A_REF):
    row = '{:>7.2f}{:>8.2f}{:>8.2f}'.format(
        width, A_REF[width], K_REF[width])
    for label in labels:
        entry = results.get(label, {}).get(width)
        if entry is None:
            row += '{:>12}{:>7}{:>7}'.format('-', '-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>12.2f}{:>+7.2f}{:>+7.2f}'.format(
            accuracy, accuracy - A_REF[width], accuracy - K_REF[width])
    print(row)
reference = sum(A_REF.values()) / len(A_REF)
k_reference = sum(K_REF.values()) / len(K_REF)
print('\n{:24} mean {:.2f}'.format('A (BX)', reference))
print('{:24} mean {:.2f}'.format('K (DD)', k_reference))
for label in labels:
    table = results.get(label, {})
    if len(table) == len(A_REF):
        mean = sum(100.0 * (1.0 - v[1]) for v in table.values()) / len(table)
        print('{:24} mean {:.2f}   vs A {:+.2f}   vs K {:+.2f}'.format(
            label, mean, mean - reference, mean - k_reference))
    else:
        print('{:24} {} of 16 widths read'.format(label, len(table)))

In [ ]:
# keep the checkpoints in the output, under a name the next session finds
print('checkpoints in', CK, sorted(os.listdir(CK)))